In [1]:
import importlib
import pandas as pd
import bdv_common as bdv

importlib.reload(bdv)            # picks up any change to the file
engine = bdv.connect_mysql()     # hidden box asks for the MySQL password

MySQL password:  ········


Connected to: ecommerce_bdv


In [12]:
display(bdv.show_status(engine))
display(bdv.show_kpis(engine).T)

,batch_id,batch_label,events_planned,events_loaded,loaded_at,summaries_done_at,graph_done_at,status
0,0,Baseline,1653660,1653660,None,2026-10-06 20:25:19,None,Baseline
1,1,Batch 1,131711,0,None,NaT,None,Waiting
2,2,Batch 2,144337,0,None,NaT,None,Waiting
3,3,Batch 3,128559,0,None,NaT,None,Waiting
4,4,Batch 4,120763,0,None,NaT,None,Waiting
5,5,Batch 5,126924,0,None,NaT,None,Waiting
6,6,Batch 6,127810,0,None,NaT,None,Waiting
7,7,Batch 7,119057,0,None,NaT,None,Waiting
8,8,Batch 8,138789,0,None,NaT,None,Waiting
9,9,Batch 9,64491,0,None,NaT,None,Waiting


,0
visitors,840006
views,1599672.0
carts,40649.0
purchases,13339.0
transactions,10563
view_to_cart_pct,2.54
cart_to_purchase_pct,32.82
data_up_to,2015-07-21 20:32:23.468000
latest_batch,0


In [13]:
bdv.run_batch(engine, 1)
display(bdv.show_status(engine))
display(bdv.show_kpis(engine).T)

=== Batch 1 ===
batch 1: 131,711 events loaded into MySQL in 78s (127,712 views, 3,042 carts, 957 purchases)
  category lookup: 131,711 rows added in 13s
  summary daily     done in 1s
  summary hourly    done in 1s
  summary product   done in 3s
  summary category  done in 1s
  summary visitor   done in 2s
batch 1: added to the summaries.
=== Batch 1 finished in 100s ===



,batch_id,batch_label,events_planned,events_loaded,loaded_at,summaries_done_at,graph_done_at,status
0,0,Baseline,1653660,1653660,NaT,2026-10-06 20:25:19,None,Baseline
1,1,Batch 1,131711,131711,2026-10-07 00:49:54,2026-10-07 00:50:16,None,Partly applied
2,2,Batch 2,144337,0,NaT,NaT,None,Waiting
3,3,Batch 3,128559,0,NaT,NaT,None,Waiting
4,4,Batch 4,120763,0,NaT,NaT,None,Waiting
5,5,Batch 5,126924,0,NaT,NaT,None,Waiting
6,6,Batch 6,127810,0,NaT,NaT,None,Waiting
7,7,Batch 7,119057,0,NaT,NaT,None,Waiting
8,8,Batch 8,138789,0,NaT,NaT,None,Waiting
9,9,Batch 9,64491,0,NaT,NaT,None,Waiting


,0
visitors,905946
views,1727384.0
carts,43691.0
purchases,14296.0
transactions,11266
view_to_cart_pct,2.53
cart_to_purchase_pct,32.72
data_up_to,2015-07-26 23:59:58.790000
latest_batch,1


In [14]:
tot = bdv.q(engine, """SELECT COUNT(*) AS events, SUM(event_type='view') AS views,
                              SUM(event_type='addtocart') AS carts, SUM(event_type='transaction') AS purchases,
                              COUNT(DISTINCT visitor_id) AS visitors FROM events""").iloc[0]
k = bdv.show_kpis(engine).iloc[0]
kpi_events = int(bdv.q(engine, "SELECT total_events AS t FROM dash_kpi_overall")["t"][0])

print("Events table  :", int(tot.events), int(tot.views), int(tot.carts), int(tot.purchases), int(tot.visitors))
print("Dashboard view:", kpi_events, int(k.views), int(k.carts), int(k.purchases), int(k.visitors))

plan = pd.read_csv("replay_plan.csv")
log = bdv.q(engine, """SELECT batch_id, events_loaded, views_loaded, carts_loaded, purchases_loaded
                       FROM replay_log WHERE batch_id > 0 AND loaded_at IS NOT NULL""")
m = log.merge(plan, on="batch_id")
print("Log matches the plan:", bool(((m.events_loaded == m.events_planned) & (m.views_loaded == m.views_planned) &
                                     (m.carts_loaded == m.carts_planned) & (m.purchases_loaded == m.purchases_planned)).all()))

Events table  : 1785371 1727384 43691 14296 905946
Dashboard view: 1785371 1727384 43691 14296 905946
Log matches the plan: True


In [15]:
bdv.show_status(engine)

,batch_id,batch_label,events_planned,events_loaded,loaded_at,summaries_done_at,graph_done_at,status
0,0,Baseline,1653660,1653660,NaT,2026-10-06 20:25:19,None,Baseline
1,1,Batch 1,131711,131711,2026-10-07 00:49:54,2026-10-07 00:50:16,None,Partly applied
2,2,Batch 2,144337,0,NaT,NaT,None,Waiting
3,3,Batch 3,128559,0,NaT,NaT,None,Waiting
4,4,Batch 4,120763,0,NaT,NaT,None,Waiting
5,5,Batch 5,126924,0,NaT,NaT,None,Waiting
6,6,Batch 6,127810,0,NaT,NaT,None,Waiting
7,7,Batch 7,119057,0,NaT,NaT,None,Waiting
8,8,Batch 8,138789,0,NaT,NaT,None,Waiting
9,9,Batch 9,64491,0,NaT,NaT,None,Waiting


In [16]:
display(bdv.show_status(engine))
display(bdv.show_kpis(engine).T)

,batch_id,batch_label,events_planned,events_loaded,loaded_at,summaries_done_at,graph_done_at,status
0,0,Baseline,1653660,1653660,NaT,2026-10-06 20:25:19,None,Baseline
1,1,Batch 1,131711,131711,2026-10-07 00:49:54,2026-10-07 00:50:16,None,Partly applied
2,2,Batch 2,144337,0,NaT,NaT,None,Waiting
3,3,Batch 3,128559,0,NaT,NaT,None,Waiting
4,4,Batch 4,120763,0,NaT,NaT,None,Waiting
5,5,Batch 5,126924,0,NaT,NaT,None,Waiting
6,6,Batch 6,127810,0,NaT,NaT,None,Waiting
7,7,Batch 7,119057,0,NaT,NaT,None,Waiting
8,8,Batch 8,138789,0,NaT,NaT,None,Waiting
9,9,Batch 9,64491,0,NaT,NaT,None,Waiting


,0
visitors,905946
views,1727384.0
carts,43691.0
purchases,14296.0
transactions,11266
view_to_cart_pct,2.53
cart_to_purchase_pct,32.72
data_up_to,2015-07-26 23:59:58.790000
latest_batch,1


In [17]:
bdv.run_batch(engine, 2)
display(bdv.show_status(engine))
display(bdv.show_kpis(engine).T)

=== Batch 2 ===
batch 2: 144,337 events loaded into MySQL in 41s (139,112 views, 3,881 carts, 1,344 purchases)
  category lookup: 144,337 rows added in 6s
  summary daily     done in 0s
  summary hourly    done in 0s
  summary product   done in 1s
  summary category  done in 1s
  summary visitor   done in 2s
batch 2: added to the summaries.
=== Batch 2 finished in 51s ===



,batch_id,batch_label,events_planned,events_loaded,loaded_at,summaries_done_at,graph_done_at,status
0,0,Baseline,1653660,1653660,NaT,2026-10-06 20:25:19,None,Baseline
1,1,Batch 1,131711,131711,2026-10-07 00:49:54,2026-10-07 00:50:16,None,Partly applied
2,2,Batch 2,144337,144337,2026-10-07 00:51:19,2026-10-07 00:51:29,None,Partly applied
3,3,Batch 3,128559,0,NaT,NaT,None,Waiting
4,4,Batch 4,120763,0,NaT,NaT,None,Waiting
5,5,Batch 5,126924,0,NaT,NaT,None,Waiting
6,6,Batch 6,127810,0,NaT,NaT,None,Waiting
7,7,Batch 7,119057,0,NaT,NaT,None,Waiting
8,8,Batch 8,138789,0,NaT,NaT,None,Waiting
9,9,Batch 9,64491,0,NaT,NaT,None,Waiting


,0
visitors,979136
views,1866496.0
carts,47572.0
purchases,15640.0
transactions,12317
view_to_cart_pct,2.55
cart_to_purchase_pct,32.88
data_up_to,2015-08-02 23:59:55.653000
latest_batch,2


In [19]:
tot = bdv.q(engine, """SELECT COUNT(*) AS events, SUM(event_type='view') AS views,
                              SUM(event_type='addtocart') AS carts, SUM(event_type='transaction') AS purchases,
                              COUNT(DISTINCT visitor_id) AS visitors FROM events""").iloc[0]
k = bdv.show_kpis(engine).iloc[0]
kpi_events = int(bdv.q(engine, "SELECT total_events AS t FROM dash_kpi_overall")["t"][0])

plan = pd.read_csv("replay_plan.csv")
log = bdv.q(engine, """SELECT batch_id, events_loaded, views_loaded, carts_loaded, purchases_loaded
                       FROM replay_log WHERE batch_id > 0 AND loaded_at IS NOT NULL""")

# expected totals = baseline + every batch that is loaded
done = plan[plan["batch_id"].isin(log["batch_id"])]
exp = (1653660 + int(done.events_planned.sum()), 1599672 + int(done.views_planned.sum()),
       40649 + int(done.carts_planned.sum()), 13339 + int(done.purchases_planned.sum()))

print("Batches loaded           :", sorted(log["batch_id"]))
print("Expected (baseline+plan) :", *exp)
print("Events table             :", int(tot.events), int(tot.views), int(tot.carts), int(tot.purchases), "| visitors", int(tot.visitors))
print("Dashboard view           :", kpi_events, int(k.views), int(k.carts), int(k.purchases), "| visitors", int(k.visitors))

m = log.merge(plan, on="batch_id")
print("Log matches the plan     :", bool(((m.events_loaded == m.events_planned) & (m.views_loaded == m.views_planned) &
                                          (m.carts_loaded == m.carts_planned) & (m.purchases_loaded == m.purchases_planned)).all()))
print("Events table = expected  :", (int(tot.events), int(tot.views), int(tot.carts), int(tot.purchases)) == exp)
print("Dashboard view = events  :", (kpi_events, int(k.views), int(k.carts), int(k.purchases), int(k.visitors)) ==
                                     (int(tot.events), int(tot.views), int(tot.carts), int(tot.purchases), int(tot.visitors)))

Batches loaded           : [1, 2]
Expected (baseline+plan) : 1929708 1866496 47572 15640
Events table             : 1929708 1866496 47572 15640 | visitors 979136
Dashboard view           : 1929708 1866496 47572 15640 | visitors 979136
Log matches the plan     : True
Events table = expected  : True
Dashboard view = events  : True


In [20]:
bdv.show_status(engine)

,batch_id,batch_label,events_planned,events_loaded,loaded_at,summaries_done_at,graph_done_at,status
0,0,Baseline,1653660,1653660,NaT,2026-10-06 20:25:19,None,Baseline
1,1,Batch 1,131711,131711,2026-10-07 00:49:54,2026-10-07 00:50:16,None,Partly applied
2,2,Batch 2,144337,144337,2026-10-07 00:51:19,2026-10-07 00:51:29,None,Partly applied
3,3,Batch 3,128559,0,NaT,NaT,None,Waiting
4,4,Batch 4,120763,0,NaT,NaT,None,Waiting
5,5,Batch 5,126924,0,NaT,NaT,None,Waiting
6,6,Batch 6,127810,0,NaT,NaT,None,Waiting
7,7,Batch 7,119057,0,NaT,NaT,None,Waiting
8,8,Batch 8,138789,0,NaT,NaT,None,Waiting
9,9,Batch 9,64491,0,NaT,NaT,None,Waiting


In [21]:
display(bdv.show_status(engine))
display(bdv.show_kpis(engine).T)

,batch_id,batch_label,events_planned,events_loaded,loaded_at,summaries_done_at,graph_done_at,status
0,0,Baseline,1653660,1653660,NaT,2026-10-06 20:25:19,None,Baseline
1,1,Batch 1,131711,131711,2026-10-07 00:49:54,2026-10-07 00:50:16,None,Partly applied
2,2,Batch 2,144337,144337,2026-10-07 00:51:19,2026-10-07 00:51:29,None,Partly applied
3,3,Batch 3,128559,0,NaT,NaT,None,Waiting
4,4,Batch 4,120763,0,NaT,NaT,None,Waiting
5,5,Batch 5,126924,0,NaT,NaT,None,Waiting
6,6,Batch 6,127810,0,NaT,NaT,None,Waiting
7,7,Batch 7,119057,0,NaT,NaT,None,Waiting
8,8,Batch 8,138789,0,NaT,NaT,None,Waiting
9,9,Batch 9,64491,0,NaT,NaT,None,Waiting


,0
visitors,979136
views,1866496.0
carts,47572.0
purchases,15640.0
transactions,12317
view_to_cart_pct,2.55
cart_to_purchase_pct,32.88
data_up_to,2015-08-02 23:59:55.653000
latest_batch,2
